# Final Project: Predicting Satellite Images with CNN and PyTorch
Student: K12443705


In [ ]:
# ============================================================
# Final Project: Predicting Satellite Images with CNN and PyTorch
# Student: K12443705
# ============================================================

from pathlib import Path
import os
import random
import time
import copy

import numpy as np
import pandas as pd

from PIL import Image

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_DIR = Path.cwd()

DATA_DIR = PROJECT_DIR / "data"
TEST_DIR = PROJECT_DIR / "public_test_data"

ASSETS_DIR = PROJECT_DIR / "assets"
PLOTS_DIR = ASSETS_DIR / "plots"
WEIGHTS_DIR = ASSETS_DIR / "weights"

APP_DIR = PROJECT_DIR / "app"

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
APP_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = WEIGHTS_DIR / "best_model.pth"
SUBMISSION_PATH = PROJECT_DIR / "submission.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project directory:", PROJECT_DIR)
print("Training data exists:", DATA_DIR.exists())
print("Public test data exists:", TEST_DIR.exists())
print("Device:", DEVICE)


In [ ]:
def preprocess(data_folder: str) -> tuple[pd.DataFrame, dict]:
    data_path = Path(data_folder)
    if not data_path.exists():
        raise FileNotFoundError(f"Data folder not found: {data_path}")

    class_names = sorted([folder.name for folder in data_path.iterdir() if folder.is_dir()])
    if len(class_names) == 0:
        raise ValueError(f"No class folders found in: {data_path}")

    class_to_label = {class_name: idx for idx, class_name in enumerate(class_names)}
    label_to_class = {idx: class_name for class_name, idx in class_to_label.items()}

    rows = []
    valid_extensions = [".jpg", ".jpeg", ".png"]

    for class_name in class_names:
        class_folder = data_path / class_name
        label = class_to_label[class_name]
        for image_path in sorted(class_folder.iterdir()):
            if image_path.suffix.lower() in valid_extensions:
                rows.append({"folder": str(class_folder), "file_name": image_path.name, "label": label})

    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError("No images found. Please check your data folder.")

    return df, label_to_class


df, label_to_class = preprocess(DATA_DIR)
class_to_label = {class_name: label for label, class_name in label_to_class.items()}
print("Number of images:", len(df))
print("Number of classes:", len(label_to_class))
print(label_to_class)
df.head()


In [ ]:
df["class_name"] = df["label"].map(label_to_class)
class_distribution = df["class_name"].value_counts().sort_index()
print(class_distribution)

train_df, val_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df["label"])
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print(train_df["class_name"].value_counts().sort_index())
print(val_df["class_name"].value_counts().sort_index())


In [ ]:
def load_image_from_row(row: pd.Series) -> Image.Image:
    image_path = Path(row["folder"]) / row["file_name"]
    return Image.open(image_path).convert("RGB")


def show_samples(df: pd.DataFrame, num_samples: int = 5) -> None:
    sample_df = df.sample(n=num_samples, random_state=SEED).reset_index(drop=True)
    fig, axes = plt.subplots(1, num_samples, figsize=(3 * num_samples, 3))
    if num_samples == 1:
        axes = [axes]
    for idx, ax in enumerate(axes):
        row = sample_df.iloc[idx]
        image = load_image_from_row(row)
        label_name = label_to_class[int(row["label"])]
        ax.imshow(image)
        ax.set_title(f"Label: {label_name}", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    save_path = PLOTS_DIR / "random_samples.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


def average_pixel_plot(df: pd.DataFrame) -> None:
    red_values, green_values, blue_values = [], [], []
    for _, row in df.iterrows():
        arr = np.array(load_image_from_row(row))
        red_values.append(arr[:, :, 0].mean())
        green_values.append(arr[:, :, 1].mean())
        blue_values.append(arr[:, :, 2].mean())
    plt.figure(figsize=(10, 6))
    plt.hist(red_values, bins=40, alpha=0.5, label="Red")
    plt.hist(green_values, bins=40, alpha=0.5, label="Green")
    plt.hist(blue_values, bins=40, alpha=0.5, label="Blue")
    plt.title("Distribution of Average Pixel Values")
    plt.xlabel("Average Pixel Value")
    plt.ylabel("Frequency")
    plt.legend()
    plt.grid(True, alpha=0.3)
    save_path = PLOTS_DIR / "average_pixel_distribution.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


def average_brightness_per_class(df: pd.DataFrame) -> None:
    brightness_rows = []
    for _, row in df.iterrows():
        arr = np.array(load_image_from_row(row))
        brightness_rows.append({"class_name": label_to_class[int(row["label"])], "brightness": arr.mean()})
    brightness_df = pd.DataFrame(brightness_rows)
    class_names = sorted(brightness_df["class_name"].unique())
    data_to_plot = [brightness_df[brightness_df["class_name"] == c]["brightness"].values for c in class_names]
    plt.figure(figsize=(12, 6))
    plt.boxplot(data_to_plot, labels=class_names, showfliers=True)
    plt.title("Average Brightness Distribution per Class")
    plt.xlabel("Class")
    plt.ylabel("Average Brightness")
    plt.xticks(rotation=45, ha="right")
    plt.grid(True, axis="y", alpha=0.3)
    save_path = PLOTS_DIR / "average_brightness.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
show_samples(train_df, num_samples=5)
average_pixel_plot(train_df)
average_brightness_per_class(train_df)


In [ ]:
class SatelliteDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None, has_labels: bool = True):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.has_labels = has_labels
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = Path(row["folder"]) / row["file_name"]
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        if self.has_labels:
            return image, int(row["label"])
        return image, row["file_name"]

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
val_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

BATCH_SIZE = 64
NUM_WORKERS = 0
train_dataset = SatelliteDataset(train_df, transform=train_transform, has_labels=True)
val_dataset = SatelliteDataset(val_df, transform=val_test_transform, has_labels=True)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


In [ ]:
class SatelliteCNN(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(256 * 4 * 4, 512), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(512, 128), nn.ReLU(inplace=True), nn.Dropout(0.3), nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

model = SatelliteCNN(num_classes=len(label_to_class)).to(DEVICE)
print(model)


In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0
    for images, labels in dataloader:
        images = images.to(device); labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward(); optimizer.step()
        _, preds = torch.max(outputs, dim=1)
        running_loss += loss.item() * images.size(0)
        running_correct += (preds == labels).sum().item()
        running_total += labels.size(0)
    return running_loss / running_total, running_correct / running_total

def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_correct = 0
    running_total = 0
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device); labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, dim=1)
            running_loss += loss.item() * images.size(0)
            running_correct += (preds == labels).sum().item()
            running_total += labels.size(0)
    return running_loss / running_total, running_correct / running_total


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

EPOCHS = 30
EARLY_STOPPING_PATIENCE = 7
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
best_model_state = copy.deepcopy(model.state_dict())
epochs_without_improvement = 0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step(val_acc)
    history['train_loss'].append(train_loss); history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss); history['val_acc'].append(val_acc)
    print(f"Epoch [{epoch + 1:02d}/{EPOCHS}] Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = copy.deepcopy(model.state_dict())
        torch.save(best_model_state, MODEL_PATH)
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print('Early stopping triggered.')
        break

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()


In [ ]:
def plot_training_curves(history: dict) -> None:
    epochs_range = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(epochs_range, history['train_loss'], label='Training Loss')
    axes[0].plot(epochs_range, history['val_loss'], label='Validation Loss')
    axes[1].plot(epochs_range, history['val_acc'], label='Validation Accuracy')
    axes[0].legend(); axes[1].legend(); axes[0].grid(alpha=0.3); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(PLOTS_DIR / 'training_curves.png', dpi=150, bbox_inches='tight'); plt.show()

plot_training_curves(history)


In [ ]:
def get_predictions(model, dataloader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device); labels = labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy()); all_labels.extend(labels.cpu().numpy())
    return np.array(all_preds), np.array(all_labels)

val_preds, val_labels = get_predictions(model, val_loader, DEVICE)
cm = confusion_matrix(val_labels, val_preds)
class_names = [label_to_class[i] for i in range(len(label_to_class))]
fig, ax = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names).plot(ax=ax, xticks_rotation=45, cmap='Blues', values_format='d')
plt.tight_layout(); plt.savefig(PLOTS_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
def show_misclassified_samples(model, dataset, device, num_samples: int = 5) -> None:
    model.eval(); misclassified = []
    with torch.no_grad():
        for idx in range(len(dataset)):
            image_tensor, true_label = dataset[idx]
            pred_label = int(torch.argmax(model(image_tensor.unsqueeze(0).to(device)), dim=1).item())
            if pred_label != int(true_label):
                misclassified.append((idx, int(true_label), pred_label))
            if len(misclassified) >= num_samples:
                break
    if len(misclassified) == 0:
        print('No misclassified samples found.'); return
    fig, axes = plt.subplots(1, len(misclassified), figsize=(4 * len(misclassified), 4))
    if len(misclassified) == 1: axes = [axes]
    for ax, (idx, true_label, pred_label) in zip(axes, misclassified):
        row = dataset.df.iloc[idx]
        image = load_image_from_row(row)
        ax.imshow(image)
        ax.set_title(f"True: {label_to_class[true_label]}\nPred: {label_to_class[pred_label]}", fontsize=9)
        ax.axis('off')
    plt.tight_layout(); plt.savefig(PLOTS_DIR / 'misclassified_samples.png', dpi=150, bbox_inches='tight'); plt.show()

show_misclassified_samples(model, val_dataset, DEVICE, num_samples=5)


In [ ]:
def preprocess_test(data_folder: str) -> pd.DataFrame:
    test_path = Path(data_folder)
    if not test_path.exists():
        raise FileNotFoundError(f"Test folder not found: {test_path}")
    valid_extensions = ['.jpg', '.jpeg', '.png']
    rows = []
    for image_path in sorted(test_path.iterdir()):
        if image_path.is_file() and image_path.suffix.lower() in valid_extensions:
            rows.append({'folder': str(test_path), 'file_name': image_path.name})
    test_df = pd.DataFrame(rows)
    if test_df.empty:
        raise ValueError('No test images found.')
    return test_df

test_df = preprocess_test(TEST_DIR)
test_df.head()


In [ ]:
test_dataset = SatelliteDataset(test_df, transform=val_test_transform, has_labels=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

def predict_test_data(model, dataloader, device, label_to_class: dict) -> pd.DataFrame:
    model.eval(); submission_rows = []
    with torch.no_grad():
        for images, file_names in dataloader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, dim=1)
            for file_name, pred_label in zip(file_names, preds.cpu().numpy()):
                submission_rows.append({'file_name': file_name, 'predicted_class': label_to_class[int(pred_label)]})
    return pd.DataFrame(submission_rows)

submission_df = predict_test_data(model, test_loader, DEVICE, label_to_class)
submission_df.to_csv(SUBMISSION_PATH, index=False, header=False)
submission_df.head()


In [ ]:
app_code = r'''# Mojtaba Akhundzadah - Student ID: K12443705
from pathlib import Path
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from shiny.express import input, render, ui

APP_DIR = Path(__file__).parent
PROJECT_DIR = APP_DIR.parent
MODEL_PATH = PROJECT_DIR / 'assets' / 'weights' / 'best_model.pth'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASS_NAMES = ['AnnualCrop','Forest','HerbaceousVegetation','Highway','Industrial','Pasture','PermanentCrop','Residential','River','SeaLake']

class SatelliteCNN(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(256*4*4,512), nn.ReLU(inplace=True), nn.Dropout(0.5), nn.Linear(512,128), nn.ReLU(inplace=True), nn.Dropout(0.3), nn.Linear(128,num_classes))
    def forward(self, x): return self.classifier(self.features(x))

prediction_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])])
model = SatelliteCNN(num_classes=len(CLASS_NAMES)).to(DEVICE)
state_dict = torch.load(MODEL_PATH, map_location=DEVICE)
_ = model.load_state_dict(state_dict)
_ = model.eval()

ui.page_opts(title='Satellite Image Classifier', fillable=True)
ui.h2('Satellite Image Classifier')
ui.input_file('image_file', 'Select satellite image', accept=['.jpg','.jpeg','.png'], multiple=False)

@render.image
def uploaded_image():
    f = input.image_file()
    if not f: return None
    return {'src': f[0]['datapath'], 'width': '300px', 'height': '300px', 'alt': 'Uploaded satellite image'}

def predict_uploaded_image():
    f = input.image_file()
    if not f: return None
    image = Image.open(f[0]['datapath']).convert('RGB')
    image_tensor = prediction_transform(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(image_tensor)
        probabilities = F.softmax(logits, dim=1).squeeze(0)
    probabilities_cpu = probabilities.cpu().numpy()
    predicted_index = int(probabilities.argmax().item())
    predicted_class = CLASS_NAMES[predicted_index]
    confidence = float(probabilities[predicted_index].item())
    probability_df = pd.DataFrame({'Class': CLASS_NAMES, 'Probability': probabilities_cpu}).sort_values(by='Probability', ascending=False).reset_index(drop=True)
    probability_df['Probability'] = probability_df['Probability'].round(4)
    return predicted_class, confidence, probability_df

@render.ui
def prediction_card():
    result = predict_uploaded_image()
    if result is None: return ui.div('Please upload an image first.')
    predicted_class, confidence, _ = result
    return ui.div(ui.h4('Prediction'), ui.p(f'Predicted class: {predicted_class}'), ui.p(f'Confidence: {confidence * 100:.2f}%'))

@render.data_frame
def probability_table():
    result = predict_uploaded_image()
    if result is None:
        return pd.DataFrame({'Class': CLASS_NAMES, 'Probability': [0.0]*len(CLASS_NAMES)})
    _, _, probability_df = result
    return probability_df
'''

app_path = APP_DIR / 'app.py'
with open(app_path, 'w', encoding='utf-8') as f:
    f.write(app_code)
print(f'Shiny app written to: {app_path}')
